# Snowflake CLI: installation and SQL queries

Use the modern **Snowflake CLI**, whose command is `snow`. This notebook contains instructions to run in **Windows PowerShell**; shell blocks are not Python cells. The Python connector notebook is separate.

Use your existing Snowflake database, schema, warehouse and role. No database or warehouse is created. CLI connections are separate sessions: they do **not** inherit the context selected in Snowsight.

The basic queries work without any orders table. Later examples reuse `ORDERS`, `ORDERS_S3_STAGE`, and optional `ExternalOrderTable` from the S3 integration notebook. AWS keys are not needed to log in to Snowflake.

## 1. Install the CLI on Windows

Open PowerShell in this notebook's directory. These commands assume the Windows Python launcher and Python 3.12 are installed. Use an available supported Python version if yours differs. The CLI's Python installation supports Python 3.10 or later; Snowflake also offers standalone installers if Python installation is inconvenient.

```powershell
py -3.12 --version
py -3.12 -m venv .venv-snow-cli
.\.venv-snow-cli\Scripts\python.exe -m pip install --upgrade pip
.\.venv-snow-cli\Scripts\python.exe -m pip install snowflake-cli
.\.venv-snow-cli\Scripts\snow.exe --version
.\.venv-snow-cli\Scripts\snow.exe --help
```

For the rest of this PowerShell session, add the environment's executables to PATH:

```powershell
$env:Path = "$(Resolve-Path '.\.venv-snow-cli\Scripts');$env:Path"
snow --version
```

Repeat the PATH line in a new terminal, or invoke the full `snow.exe` path. No activation-policy change is needed. Install package **snowflake-cli**, not `snowflake` or the older `snowflake-cli-labs` name. These steps install only when you run them.

[Official CLI installation](https://docs.snowflake.com/en/developer-guide/snowflake-cli/installation/installation)

## 2. Collect your connection details

In your working Snowsight worksheet, run:

```sql
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(),
       CURRENT_DATABASE(), CURRENT_SCHEMA();
```

Copy the existing context names. Obtain the **account identifier** from Snowflake account details, typically `organization-account`; do not paste the full Snowsight URL into the account field.

Choose the authentication method your user actually supports:

| Account setup | Authentication |
|---|---|
| Password login with enrolled MFA | `snowflake`; complete required MFA challenges |
| Corporate SAML SSO with a local browser | `externalbrowser`; password is unnecessary |
| Password login prohibited / unattended automation | Ask the administrator for an approved PAT or key-pair method |

Browser SSO requires account-side SSO configuration; it is not automatically available merely because Snowsight opens in a browser. Do not disable MFA for this lab.

## 3. Save a simple connection

```powershell
snow connection add
```

Enter connection name `classroom`, your account, user, existing role, warehouse, database and schema. Leave the password blank; provide it through the environment in section 4. Choose `snowflake` or `externalbrowser` as appropriate. Leave optional host, port, region and key/token paths blank unless your administrator supplies them.

```powershell
snow connection list
```

The command reports where it writes configuration. Keep that file private. If you already use `connections.toml`, CLI reads connections there instead of those in `config.toml`; avoid duplicate definitions across files.

[Connection configuration](https://docs.snowflake.com/en/developer-guide/snowflake-cli/connecting/configure-connections)

## 4. Supply a password privately, then test

Skip the password lines for `externalbrowser`. For password authentication, run in PowerShell; the prompt hides what you type and keeps the literal password out of command history:

```powershell
$snowPassword = Read-Host 'Snowflake password' -AsSecureString
$env:SNOWFLAKE_CONNECTIONS_CLASSROOM_PASSWORD = [System.Net.NetworkCredential]::new('', $snowPassword).Password
Remove-Variable snowPassword
snow connection test -c classroom
```

Complete MFA as requested. If your enrolled method uses a one-time code, consult `snow connection test --help` for the `--mfa-passcode` option and supply a fresh code for that connection. A successful test should report `OK`.

Each CLI invocation can require authentication again. MFA caching depends on account settings; this lab does not change them.

The password remains in this terminal's process environment until removed or the terminal closes. Do not print it. [CLI authentication](https://docs.snowflake.com/en/developer-guide/snowflake-cli/connecting/configure-connections)

## 5. Run your first queries

```powershell
snow sql -c classroom -q "SELECT CURRENT_VERSION(), CURRENT_USER();"
snow sql -c classroom -q "SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(), CURRENT_ROLE();"
snow sql -c classroom -q "SELECT 1001 AS ORDER_ID, 'NEW' AS STATUS, 120.50 AS ORDER_TOTAL;"
```

If a context value is missing, update your connection or override it for that command using your existing names:

```powershell
snow sql -c classroom --database YOUR_DATABASE --schema YOUR_SCHEMA --warehouse YOUR_WAREHOUSE -q "SELECT CURRENT_DATABASE(), CURRENT_SCHEMA();"
```

A `USE SCHEMA` executed in one CLI process does not configure a later CLI process. Put related statements in one SQL file/session, or keep the context in the connection.

[SQL command options](https://docs.snowflake.com/en/developer-guide/snowflake-cli/command-reference/sql-commands/sql)

## 6. Query orders from the S3 exercise

Run these only after the corresponding objects exist in your selected schema:

```powershell
snow sql -c classroom -q "SELECT * FROM ORDERS ORDER BY ORDER_ID LIMIT 20;"
snow sql -c classroom -q "SELECT STATUS, COUNT(*) AS ORDER_COUNT, SUM(ORDER_TOTAL) AS TOTAL_AMOUNT FROM ORDERS GROUP BY STATUS ORDER BY STATUS;"
snow sql -c classroom -q "LIST @ORDERS_S3_STAGE;"
snow sql -c classroom -q "SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@ORDERS_S3_STAGE) ORDER BY RELATIVE_PATH;"
snow sql -c classroom -q "SELECT ORDER_ID, STATUS, ORDER_TOTAL FROM ExternalOrderTable ORDER BY ORDER_ID LIMIT 20;"
```

The final command requires the optional external table. S3 operations use credentials already stored on the stage; expired sandbox credentials can break stage access while a native `ORDERS` query still works.

## 7. Execute a SQL file

The companion `orders_queries.sql` contains a context check and two read-only orders queries. Run from its directory:

```powershell
snow sql -c classroom -f .\orders_queries.sql
if ($LASTEXITCODE -ne 0) { throw 'Snowflake SQL execution failed.' }
```

To inspect statements before execution:

```powershell
Get-Content -LiteralPath .\orders_queries.sql
```

SQL files are also useful for keeping multiple statements in one session. Do not place passwords or AWS secrets in a shared SQL file.

[Execute SQL files](https://docs.snowflake.com/en/developer-guide/snowflake-cli/sql/execute-sql)

## 8. Export a query result

```powershell
snow sql -c classroom --format csv -q "SELECT ORDER_ID, STATUS, ORDER_TOTAL FROM ORDERS ORDER BY ORDER_ID;" > orders_cli_export.csv
if ($LASTEXITCODE -ne 0) { throw 'Export failed; inspect the CLI error.' }
```

Use a single query for a simple CSV export. Check the generated file before sharing it. `--format json` is another option. PowerShell controls redirected text encoding; if you need a specific encoding for another application, use `Out-File -Encoding utf8` explicitly.

[SQL output options](https://docs.snowflake.com/en/developer-guide/snowflake-cli/command-reference/sql-commands/sql)

## 9. Common problems and finish

| Problem | Action |
|---|---|
| `snow` is not recognized | Use the environment's full executable path or repeat the PATH line |
| Account/hostname error | Use the connection account identifier, not the Snowsight URL |
| Login fails | Verify the authentication method, current password and MFA enrollment |
| Browser SSO fails | Confirm SSO is configured and this machine can open the browser |
| Object does not exist or not authorized | Compare database/schema/role with the working Snowsight worksheet |
| No warehouse selected | Set an existing permitted warehouse in the connection |
| TLS/network error | Use the organization's approved proxy/allowlist settings; do not disable certificate validation |

Remove the password from the current terminal after class:

```powershell
Remove-Item Env:\SNOWFLAKE_CONNECTIONS_CLASSROOM_PASSWORD -ErrorAction SilentlyContinue
```

No Snowflake object cleanup is needed: this notebook only queries existing objects. Installation instructions and command syntax were reviewed; live authentication must be verified in your account.